In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.ecommerce.silver_volume
""")

mlflow.set_tracking_uri("databricks")

import os
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/ecommerce/silver_volume/mlflow_tmp"

In [0]:
# Spark ML imports
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# MLflow imports
import mlflow
import mlflow.spark

# Other utilities
import os

In [0]:
import mlflow

experiments = mlflow.search_experiments()

for exp in experiments:
    print(exp.name, " | ID:", exp.experiment_id)

In [0]:
experiment_name = "/Users/abhinavjajoo19@gmail.com/day7_mlflow_experiment"
mlflow.set_experiment(experiment_name)

In [0]:
df = spark.read.table("workspace.ecommerce.train_dataset")

In [0]:
feature_cols = [
    "total_events",
    "unique_products",
    "avg_price"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(df).select("features", "label")

In [0]:
train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

In [0]:
for reg in [0.0, 0.01, 0.1]:

    with mlflow.start_run(run_name=f"LR_reg_{reg}"):

        lr = LogisticRegression(
            featuresCol="features",
            labelCol="label",
            regParam=reg
        )

        model = lr.fit(train_df)
        pred = model.transform(test_df)
        auc = evaluator.evaluate(pred)

        # Log params
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("regParam", reg)

        # Log metric
        mlflow.log_metric("AUC", auc)

        # IMPORTANT — log model with UC path support
        mlflow.spark.log_model(
            model,
            artifact_path="model"
        )

        print(f"Logged LR reg={reg}, AUC={auc}")

In [0]:
for trees in [20, 50]:
    for depth in [5, 10]:

        with mlflow.start_run(run_name=f"RF_{trees}_trees_depth_{depth}"):

            rf = RandomForestClassifier(
                featuresCol="features",
                labelCol="label",
                numTrees=trees,
                maxDepth=depth
            )

            model = rf.fit(train_df)
            pred = model.transform(test_df)
            auc = evaluator.evaluate(pred)

            mlflow.log_param("model_type", "RandomForest")
            mlflow.log_param("numTrees", trees)
            mlflow.log_param("maxDepth", depth)

            mlflow.log_metric("AUC", auc)

            mlflow.spark.log_model(
                model,
                artifact_path="model"
            )

            print(f"Logged RF trees={trees}, depth={depth}, AUC={auc}")

In [0]:
# ==============================
# DAY 8 – Batch Inference Setup
# ==============================

import mlflow
import mlflow.spark
import os
from pyspark.sql import functions as F

# Required for serverless clusters
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/workspace/ecommerce/silver_volume/mlflow_tmp"

# Your Day-7 experiment
experiment_name = "/Users/abhinavjajoo19@gmail.com/day7_mlflow_experiment"
mlflow.set_experiment(experiment_name)

# Tables & Paths
silver_table = "workspace.ecommerce.user_features_silver"
gold_table = "workspace.ecommerce.user_predictions_gold"

print("Setup completed")

In [0]:
# ==============================
# Load Silver Features
# ==============================

features_df = spark.table(silver_table)

print("Total users to score:", features_df.count())
display(features_df.limit(5))

In [0]:
# ==============================
# Get Best Model from MLflow
# ==============================

runs = mlflow.search_runs(order_by=["metrics.AUC DESC"])

if runs.empty:
    raise Exception("No MLflow runs found. Run Day 7 first.")

best_run = runs.iloc[0]
run_id = best_run.run_id

print("Best Run ID:", run_id)
print("Best AUC:", best_run["metrics.AUC"])

In [0]:
import mlflow
import mlflow.spark
from pyspark.sql import functions as F

# ==============================
# Paths & Tables
# ==============================

experiment_path = "/Users/abhinavjajoo19@gmail.com/day7_mlflow_experiment"

silver_table = "workspace.ecommerce.user_features_silver"
gold_table = "workspace.ecommerce.user_predictions_gold"

gold_path = "/Volumes/workspace/ecommerce/silver_volume/gold_predictions"

In [0]:
# Set experiment
mlflow.set_experiment(experiment_path)

# Get all runs sorted by AUC
runs = mlflow.search_runs(order_by=["metrics.AUC DESC"])

if runs.empty:
    raise Exception("No MLflow runs found. Run Day 7 first.")

# Keep only runs that have model artifact
runs_with_model = runs[runs["artifact_uri"].notna()]

best_run = runs_with_model.iloc[0]
run_id = best_run.run_id

print("Best Run ID:", run_id)
print("Best AUC:", best_run["metrics.AUC"])

In [0]:
# Set experiment
mlflow.set_experiment(experiment_path)

# Get all runs sorted by AUC
runs = mlflow.search_runs(order_by=["metrics.AUC DESC"])

if runs.empty:
    raise Exception("No MLflow runs found. Run Day 7 first.")

# Keep only runs that have model artifact
runs_with_model = runs[runs["artifact_uri"].notna()]

best_run = runs_with_model.iloc[0]
run_id = best_run.run_id

print("Best Run ID:", run_id)
print("Best AUC:", best_run["metrics.AUC"])

In [0]:
import mlflow

experiment_path = "/Users/abhinavjajoo19@gmail.com/day7_mlflow_experiment"
mlflow.set_experiment(experiment_path)

runs = mlflow.search_runs(order_by=["metrics.AUC DESC"])

if runs.empty:
    raise Exception("No runs found")

# Keep only runs where model exists
runs_with_model = runs[runs["artifact_uri"].str.contains("mlflow", na=False)]

print("Runs with model:", len(runs_with_model))

best_run = runs_with_model.iloc[0]
run_id = best_run.run_id

print("Using Run ID:", run_id)
print("AUC:", best_run["metrics.AUC"])

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col, desc
import mlflow
import mlflow.spark

In [0]:
dbutils.fs.mkdirs("/Volumes/workspace/ecommerce/silver_volume/mlflow_tmp")

In [0]:
from pyspark.ml.feature import VectorAssembler

# Load training table from Day 5
df = spark.table("workspace.ecommerce.train_dataset")

In [0]:
feature_cols = [
    "total_events",
    "total_purchases",
    "total_spent",
    "avg_price",
    "unique_products"
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

df_ml = assembler.transform(df)

In [0]:
df_ml.select("user_id", "features", "label") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.train_dataset_ml")

In [0]:
test_df = spark.table("workspace.ecommerce.test_dataset")

test_ml = assembler.transform(test_df)

test_ml.select("user_id", "features", "label") \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.ecommerce.test_dataset_ml")

In [0]:
# ============================================
# DAY 6 → DAY 8 FULL PIPELINE (ONE RUN)
# ============================================

# from pyspark.sql import functions as F
# from pyspark.ml.feature import VectorAssembler
# from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
# from pyspark.ml.evaluation import BinaryClassificationEvaluator
# import mlflow
# import mlflow.spark

# ============================================
# CONFIG
# ============================================

catalog_db = "workspace.ecommerce"

silver_table = f"{catalog_db}.user_features_silver"
train_table = f"{catalog_db}.train_dataset"
test_table = f"{catalog_db}.test_dataset"

train_ml_table = f"{catalog_db}.train_dataset_ml"
test_ml_table = f"{catalog_db}.test_dataset_ml"

gold_table = f"{catalog_db}.user_predictions_gold"

experiment_path = "/Users/abhinavjajoo19@gmail.com/day7_mlflow_experiment"

# UC temp path (REQUIRED for Serverless MLflow)
dfs_tmp = "/Volumes/workspace/ecommerce/silver_volume/mlflow_tmp"

dbutils.fs.mkdirs(dfs_tmp)

mlflow.set_experiment(experiment_path)

# ============================================
# DAY 6 — PREPARE ML DATA (VectorAssembler)
# ============================================

print("Preparing ML datasets...")

feature_cols = [
    "total_events",
    "total_purchases",
    "total_spent",
    "avg_price",
    "unique_products"
]

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

# Load train & test
train_df = spark.table(train_table)
test_df = spark.table(test_table)

train_ml = assembler.transform(train_df).select("user_id", "features", "label")
test_ml = assembler.transform(test_df).select("user_id", "features", "label")

# Save ML tables
train_ml.write.format("delta").mode("overwrite").saveAsTable(train_ml_table)
test_ml.write.format("delta").mode("overwrite").saveAsTable(test_ml_table)

print("ML feature tables created.")

# Reload
train_ml = spark.table(train_ml_table)
test_ml = spark.table(test_ml_table)

# ============================================
# DAY 6 — MODEL TRAINING
# ============================================

print("Training models...")

evaluator = BinaryClassificationEvaluator(labelCol="label")

# -------------------------
# Logistic Regression
# -------------------------
lr_params = [0.0, 0.01, 0.1]
best_lr_auc = 0
best_lr_model = None

for reg in lr_params:
    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        regParam=reg
    )
    
    model = lr.fit(train_ml)
    pred = model.transform(test_ml)
    auc = evaluator.evaluate(pred)
    
    print(f"LR regParam={reg} → AUC={auc}")
    
    if auc > best_lr_auc:
        best_lr_auc = auc
        best_lr_model = model
        best_lr_param = reg

# -------------------------
# Random Forest
# -------------------------
rf_configs = [(20,5), (50,5), (50,10)]
best_rf_auc = 0
best_rf_model = None

for trees, depth in rf_configs:
    rf = RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=trees,
        maxDepth=depth
    )
    
    model = rf.fit(train_ml)
    pred = model.transform(test_ml)
    auc = evaluator.evaluate(pred)
    
    print(f"RF trees={trees}, depth={depth} → AUC={auc}")
    
    if auc > best_rf_auc:
        best_rf_auc = auc
        best_rf_model = model
        best_rf_param = (trees, depth)

# Choose best model
if best_rf_auc > best_lr_auc:
    final_model = best_rf_model
    model_type = "RandomForest"
    best_auc = best_rf_auc
    params = best_rf_param
else:
    final_model = best_lr_model
    model_type = "LogisticRegression"
    best_auc = best_lr_auc
    params = best_lr_param

print("\nBest Model:", model_type)
print("Best AUC:", best_auc)

# ============================================
# DAY 7 — MLflow Logging
# ============================================

print("\nLogging best model to MLflow...")

with mlflow.start_run(run_name=f"Best_{model_type}"):

    mlflow.log_param("model_type", model_type)
    mlflow.log_param("params", str(params))
    mlflow.log_metric("AUC", best_auc)

    mlflow.spark.log_model(
        final_model,
        "model",
        dfs_tmpdir=dfs_tmp
    )

print("Model logged successfully.")

#from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

# ============================================
# Score all users
# ============================================

silver_df = spark.table("workspace.ecommerce.user_features_silver")

feature_cols = [
    "total_events",
    "total_purchases",
    "total_spent",
    "avg_price",
    "unique_products"
]

# from pyspark.ml.feature import VectorAssembler
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")

silver_ml = assembler.transform(silver_df)

# Use your trained model
predictions = final_model.transform(silver_ml)

# ============================================
# Extract probability correctly (Important Fix)
# ============================================

result_df = (
    predictions
    .withColumn("purchase_probability", vector_to_array("probability")[1])
    .select(
        "user_id",
        "purchase_probability"
    )
)

# ============================================
# Save Gold Table
# ============================================

gold_table = "workspace.ecommerce.user_predictions_gold"

result_df.write.format("delta").mode("overwrite").saveAsTable(gold_table)

print("Gold prediction table created:", gold_table)

# ============================================
# Top Buyers
# ============================================

spark.sql(f"""
SELECT *
FROM {gold_table}
ORDER BY purchase_probability DESC
LIMIT 10
""").display()

In [0]:
spark.sql("""
SELECT 
  MIN(purchase_probability) AS min_prob,
  MAX(purchase_probability) AS max_prob,
  AVG(purchase_probability) AS avg_prob
FROM workspace.ecommerce.user_predictions_gold
""").show()

In [0]:
spark.sql("""
SELECT 
  CASE 
    WHEN purchase_probability > 0.8 THEN 'High Intent'
    WHEN purchase_probability > 0.3 THEN 'Medium Intent'
    ELSE 'Low Intent'
  END AS user_segment,
  COUNT(*) AS users
FROM workspace.ecommerce.user_predictions_gold
GROUP BY user_segment
ORDER BY users DESC
""").display()